# Archive Evolution Guidance Test

This notebook compares three evolution-guidance variants on the same prompt:

- online evolution guidance
- archive-based evolution guidance
- archive-based evolution guidance with bad-sample repulsion

It reuses the repo's `evo_steering_playground.py` helpers so the notebook stays close to the branch implementation.

## Colab flow

1. Run the next cell to clone the repo and checkout the branch you want to test.
2. Run the install cell.
3. Then run the rest of the notebook normally.

If you already opened this notebook from a cloned repo in Colab, the setup cell is still safe to rerun.

In [1]:
from pathlib import Path
import os
import subprocess

REPO_URL = "https://github.com/khoing-ml/Fk-Diffusion-Steering.git"
REPO_DIR = Path("/content/Fk-Diffusion-Steering")
BRANCH = "evov1"  # change this if you want to test a different branch

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    print(f"Repo already exists at {REPO_DIR}")

subprocess.run(["git", "fetch", "origin"], cwd=REPO_DIR, check=True)
subprocess.run(["git", "checkout", BRANCH], cwd=REPO_DIR, check=True)

try:
    subprocess.run(["git", "pull", "--ff-only", "origin", BRANCH], cwd=REPO_DIR, check=True)
except subprocess.CalledProcessError as exc:
    print(f"Warning: git pull failed: {exc}")

os.chdir(REPO_DIR)
print(f"cwd = {Path.cwd()}")

Repo already exists at /content/Fk-Diffusion-Steering


CalledProcessError: Command '['git', 'checkout', 'evov1']' returned non-zero exit status 1.

In [ ]:
%%bash
set -e
cd /content/Fk-Diffusion-Steering
python -m pip install -q -r requirements.txt

In [4]:
from __future__ import annotations

import json
import os
import sys
from collections import defaultdict
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import display


DEFAULT_REPO_ROOTS = [
    Path(os.environ.get("FKD_REPO_ROOT", "/content/Fk-Diffusion-Steering")),
    Path.cwd(),
]


def find_repo_root(start_paths: list[Path] | None = None) -> Path:
    start_paths = DEFAULT_REPO_ROOTS if start_paths is None else start_paths
    for start in start_paths:
        start = start.resolve()
        for candidate in [start, *start.parents]:
            if (candidate / "text_to_image" / "fkd_diffusers").exists():
                return candidate
    raise FileNotFoundError("Could not find repo root from current working directory.")


REPO_ROOT = find_repo_root()
os.chdir(REPO_ROOT)
TEXT_TO_IMAGE_ROOT = REPO_ROOT / "text_to_image"
FKD_ROOT = TEXT_TO_IMAGE_ROOT / "fkd_diffusers"

for path in [REPO_ROOT, TEXT_TO_IMAGE_ROOT, FKD_ROOT]:
    path_str = str(path)
    if path_str not in sys.path:
        sys.path.insert(0, path_str)

print(f"REPO_ROOT = {REPO_ROOT}")
print(f"TEXT_TO_IMAGE_ROOT = {TEXT_TO_IMAGE_ROOT}")

REPO_ROOT = /content/Fk-Diffusion-Steering
TEXT_TO_IMAGE_ROOT = /content/Fk-Diffusion-Steering/text_to_image


In [5]:
from evo_steering_playground import (
    ensure_dependency_compat,
    resolve_guidance_reward_fn,
    run_mode,
)
from fks_utils import do_eval, get_model

IN_COLAB = "google.colab" in sys.modules
AUTO_FIX_DEPS = IN_COLAB
ensure_dependency_compat(auto_fix=AUTO_FIX_DEPS)

ModuleNotFoundError: No module named 'evo_steering_playground'

In [ ]:
MODEL_NAME = "stable-diffusion-2-1"
PROMPT = "a cinematic photo of a corgi astronaut on mars, ultra detailed"
DEVICE = "auto"

NUM_PARTICLES = 4
TIME_STEPS = 50
LMBDA = 2.0

GUIDANCE_REWARD_FN = "ImageReward"
FALLBACK_REWARD_FN = "Clip-Score"

SVGD_STEP_SIZE = 0.12
SVGD_SIGMA = None
GUIDANCE_FREQUENCY = 5

RESAMPLE_STRATEGY = "multinomial"
EVOLUTION_RESAMPLE_STRATEGY = "none"

ARCHIVE_SIZE = 64
ARCHIVE_GOOD_QUANTILE = 0.75
ARCHIVE_BAD_QUANTILE = 0.25
ARCHIVE_BURN_IN_STEPS = 3
MIN_GOOD_ANCHORS = 8
MIN_BAD_ANCHORS = 8

SINGLE_SEED = 7
SWEEP_SEEDS = [1, 2, 3, 4, 5]

OUTPUT_DIR = TEXT_TO_IMAGE_ROOT / "output" / f"archive_notebook_{datetime.now().strftime('%Y%m%d-%H%M%S')}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

resolved_reward_fn = resolve_guidance_reward_fn(
    reward_fn=GUIDANCE_REWARD_FN,
    fallback_reward_fn=FALLBACK_REWARD_FN,
    allow_fallback=True,
)
runtime_device = "cuda" if DEVICE == "auto" and torch.cuda.is_available() else ("cpu" if DEVICE == "auto" else DEVICE)

print(json.dumps({
    "model": MODEL_NAME,
    "prompt": PROMPT,
    "device": runtime_device,
    "reward_fn": resolved_reward_fn,
    "guidance_frequency": GUIDANCE_FREQUENCY,
    "archive_burn_in_steps": ARCHIVE_BURN_IN_STEPS,
    "archive_size": ARCHIVE_SIZE,
}, indent=2))

In [ ]:
pipe = get_model(MODEL_NAME).to(runtime_device)
pipe

In [ ]:
COMMON_RUN_ARGS = {
    "prompt": PROMPT,
    "num_particles": NUM_PARTICLES,
    "time_steps": TIME_STEPS,
    "lmbda": LMBDA,
    "svgd_step_size": SVGD_STEP_SIZE,
    "svgd_sigma": SVGD_SIGMA,
    "guidance_frequency": GUIDANCE_FREQUENCY,
    "guidance_reward_fn": resolved_reward_fn,
    "resample_strategy": RESAMPLE_STRATEGY,
    "evolution_resample_strategy": EVOLUTION_RESAMPLE_STRATEGY,
    "use_anchor_archive": False,
    "archive_size": ARCHIVE_SIZE,
    "archive_good_quantile": ARCHIVE_GOOD_QUANTILE,
    "archive_bad_quantile": ARCHIVE_BAD_QUANTILE,
    "archive_burn_in_steps": ARCHIVE_BURN_IN_STEPS,
    "min_good_anchors": MIN_GOOD_ANCHORS,
    "min_bad_anchors": MIN_BAD_ANCHORS,
    "bad_guidance_strength": 0.0,
}

VARIANTS = {
    "online_evolution": {
        "mode": "evolution",
        "use_anchor_archive": False,
        "bad_guidance_strength": 0.0,
        "min_bad_anchors": 0,
    },
    "archive_evolution": {
        "mode": "evolution",
        "use_anchor_archive": True,
        "bad_guidance_strength": 0.0,
        "min_bad_anchors": 0,
    },
    "archive_contrastive": {
        "mode": "evolution",
        "use_anchor_archive": True,
        "bad_guidance_strength": 0.30,
        "min_bad_anchors": MIN_BAD_ANCHORS,
    },
}


def run_variant(*, label: str, seed: int, **overrides):
    cfg = dict(COMMON_RUN_ARGS)
    cfg.update(overrides)
    images, rewards, fkd_args = run_mode(
        pipe=pipe,
        do_eval=do_eval,
        seed=seed,
        **cfg,
    )
    return {
        "label": label,
        "seed": seed,
        "images": images,
        "rewards": rewards,
        "fkd_args": fkd_args,
        "mean": float(np.mean(rewards)),
        "std": float(np.std(rewards)),
        "best": float(np.max(rewards)),
    }


def display_result_grid(result, *, max_images: int | None = None):
    images = result["images"]
    rewards = result["rewards"]
    if max_images is not None:
        images = images[:max_images]
        rewards = rewards[:max_images]

    n = len(images)
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 4), dpi=150)
    if n == 1:
        axes = [axes]

    for idx, (ax, image, reward) in enumerate(zip(axes, images, rewards), start=1):
        ax.imshow(image)
        ax.set_title(f"#{idx} score={reward:.3f}")
        ax.axis("off")

    fig.suptitle(
        f"{result['label']} | seed={result['seed']} | "
        f"mean={result['mean']:.4f} | best={result['best']:.4f}"
    )
    fig.tight_layout()
    plt.show()


def save_result_grid(result):
    out_path = OUTPUT_DIR / f"{result['label']}_seed{result['seed']}.png"
    images = result["images"]
    rewards = result["rewards"]

    fig, axes = plt.subplots(1, len(images), figsize=(4 * len(images), 4), dpi=150)
    if len(images) == 1:
        axes = [axes]
    for idx, (ax, image, reward) in enumerate(zip(axes, images, rewards), start=1):
        ax.imshow(image)
        ax.set_title(f"#{idx} score={reward:.3f}")
        ax.axis("off")
    fig.suptitle(result["label"])
    fig.tight_layout()
    fig.savefig(out_path)
    plt.close(fig)
    return out_path


def summarize_rows(rows):
    grouped = defaultdict(list)
    for row in rows:
        grouped[row["label"]].append(row)

    summary = []
    for label, vals in grouped.items():
        mean_vals = np.array([v["mean"] for v in vals], dtype=np.float32)
        best_vals = np.array([v["best"] for v in vals], dtype=np.float32)
        summary.append({
            "label": label,
            "n_seeds": len(vals),
            "mean_of_mean": float(mean_vals.mean()),
            "std_of_mean": float(mean_vals.std()),
            "mean_of_best": float(best_vals.mean()),
        })
    summary.sort(key=lambda row: row["mean_of_mean"], reverse=True)
    return summary

## Single-seed comparison

This cell runs each variant on the same seed so you can inspect the images directly.

In [ ]:
single_seed_results = {}

for label, overrides in VARIANTS.items():
    result = run_variant(label=label, seed=SINGLE_SEED, **overrides)
    single_seed_results[label] = result
    display_result_grid(result)
    print(json.dumps(result["fkd_args"], indent=2, default=str))
    print(f"saved to: {save_result_grid(result)}")
    print("-" * 80)

## Multi-seed sweep

This is the quickest way to see whether archive guidance is more stable than online guidance.

In [ ]:
sweep_rows = []

for label, overrides in VARIANTS.items():
    for seed in SWEEP_SEEDS:
        result = run_variant(label=label, seed=seed, **overrides)
        sweep_rows.append({
            "label": label,
            "seed": seed,
            "mean": result["mean"],
            "std": result["std"],
            "best": result["best"],
        })

summary_rows = summarize_rows(sweep_rows)
summary_rows

In [ ]:
labels = [row["label"] for row in summary_rows]
mean_of_mean = [row["mean_of_mean"] for row in summary_rows]
std_of_mean = [row["std_of_mean"] for row in summary_rows]
mean_of_best = [row["mean_of_best"] for row in summary_rows]

x = np.arange(len(labels))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5), dpi=150)
ax.bar(x - width / 2, mean_of_mean, width, yerr=std_of_mean, label="mean of mean")
ax.bar(x + width / 2, mean_of_best, width, label="mean of best")
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=15)
ax.set_ylabel(resolved_reward_fn)
ax.set_title("Archive evolution sweep summary")
ax.legend()
plt.tight_layout()
plt.show()

## Suggested follow-up experiments

Try changing one knob at a time:

- `GUIDANCE_FREQUENCY`
- `ARCHIVE_BURN_IN_STEPS`
- `ARCHIVE_SIZE`
- `ARCHIVE_GOOD_QUANTILE` and `ARCHIVE_BAD_QUANTILE`
- `bad_guidance_strength` in `VARIANTS["archive_contrastive"]`
- `EVOLUTION_RESAMPLE_STRATEGY`

If the archive version still collapses, the next thing to add is archive diagnostics per step: unique-anchor count, reward histogram, and pairwise diversity.